<a href="https://colab.research.google.com/github/SAHILJAIN06/SepGuard-Sepsis-Prediction/blob/main/Sepsis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving archive.zip to archive.zip


In [ ]:
import zipfile
import os

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")

print("Files:", os.listdir("data"))

Files: ['training_setA', 'utility_sepsis_diagram.svg', 'utility_nonsepsis_diagram.svg', 'LICENSE.txt', 'training_setB', 'SHA256SUMS.txt', 'Dataset.csv', 'physionet_challenge_2019_ccm_manuscript.pdf']


In [ ]:
csv_files = []

for root, dirs, files in os.walk("data"):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print(csv_files)

['data/Dataset.csv']


In [ ]:
import pandas as pd

df = pd.read_csv(csv_files[0])

print("Shape:", df.shape)
df.head()

Shape: (1552210, 44)


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [ ]:
# Missing values
missing = df.isnull().mean().sort_values(ascending=False)
print("Top Missing:\n", missing.head(10))

# Class distribution
print("\nClass Distribution:\n", df['SepsisLabel'].value_counts(normalize=True))

Top Missing:
 Bilirubin_direct    0.998074
Fibrinogen          0.993402
TroponinI           0.990477
Bilirubin_total     0.985092
Alkalinephos        0.983932
AST                 0.983776
Lactate             0.973299
PTT                 0.970559
SaO2                0.965494
EtCO2               0.962868
dtype: float64

Class Distribution:
 SepsisLabel
0    0.982015
1    0.017985
Name: proportion, dtype: float64


In [ ]:
class DecisionAgent:
    def __init__(self, report):
        self.report = report
        self.decisions = {}

    def decide(self):
        for col in self.report.index:
            decisions = []

            missing = self.report.loc[col, "Missing_Ratio"]
            skew = self.report.loc[col, "Skewness"]
            outlier = self.report.loc[col, "Outlier_Ratio"]

            # Missing value decision
            if missing > 0.9:
                decisions.append("DROP")
            elif missing > 0.4:
                decisions.append("ADVANCED_IMPUTE")
            elif missing > 0:
                decisions.append("MEAN_IMPUTE")

            # Skewness
            if abs(skew) > 1:
                decisions.append("LOG_TRANSFORM")

            # Outliers
            if outlier > 0.1:
                decisions.append("ROBUST_SCALE")

            self.decisions[col] = decisions

        return self.decisions

In [ ]:
import pandas as pd
import numpy as np

class DataProfiler:
    def __init__(self, df):
        self.df = df
        self.report = None

    def missing_ratio(self):
        return self.df.isnull().mean()

    def skewness(self):
        return self.df.skew(numeric_only=True)

    def outlier_ratio(self):
        outliers = {}

        for col in self.df.select_dtypes(include=np.number).columns:
            series = self.df[col].dropna()

            if len(series) == 0:
                outliers[col] = np.nan
                continue

            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers[col] = ((series < lower) | (series > upper)).mean()

        return pd.Series(outliers)

    def imbalance_ratio(self, target_col):
        return self.df[target_col].value_counts(normalize=True)

    def generate_report(self, target_col=None):
        report = pd.DataFrame({
            "Missing_Ratio": self.missing_ratio(),
            "Skewness": self.skewness(),
            "Outlier_Ratio": self.outlier_ratio()
        })

        if target_col:
            print("\nClass Distribution:\n", self.imbalance_ratio(target_col))

        self.report = report
        return report

In [ ]:
profiler = DataProfiler(df)

report = profiler.generate_report(target_col="SepsisLabel")

report.head()


Class Distribution:
 SepsisLabel
0    0.982015
1    0.017985
Name: proportion, dtype: float64


,Missing_Ratio,Skewness,Outlier_Ratio
Unnamed: 0,0.000000,4.093620,0.044122
Hour,0.000000,4.093620,0.044122
HR,0.098826,0.431251,0.010031
O2Sat,0.130611,-4.152269,0.018364
Temp,0.661627,-0.329187,0.012507


In [ ]:
agent = DecisionAgent(report)
decisions = agent.decide()

for k, v in list(decisions.items())[:10]:
    print(k, "→", v)

Unnamed: 0 → ['LOG_TRANSFORM']
Hour → ['LOG_TRANSFORM']
HR → ['MEAN_IMPUTE']
O2Sat → ['MEAN_IMPUTE', 'LOG_TRANSFORM']
Temp → ['ADVANCED_IMPUTE']
SBP → ['MEAN_IMPUTE']
MAP → ['MEAN_IMPUTE', 'LOG_TRANSFORM']
DBP → ['MEAN_IMPUTE', 'LOG_TRANSFORM']
Resp → ['MEAN_IMPUTE']
EtCO2 → ['DROP']


In [ ]:
class DecisionAgent:
    def __init__(self, report):
        self.report = report
        self.decisions = {}

    def decide(self):
        for col in self.report.index:
            decisions = []

            missing = self.report.loc[col, "Missing_Ratio"]
            skew = self.report.loc[col, "Skewness"]
            outlier = self.report.loc[col, "Outlier_Ratio"]

            if col == "SepsisLabel":
                self.decisions[col] = ["KEEP"]
                continue

            # 🚨 Special Rules
            if col.lower().startswith("unnamed"):
                self.decisions[col] = ["DROP"]
                continue

            if col == "Hour":
                self.decisions[col] = ["KEEP"]
                continue

            # Missing value decision
            if missing > 0.9:
                decisions.append("DROP")
            elif missing > 0.4:
                decisions.append("ADVANCED_IMPUTE")
            elif missing > 0:
                decisions.append("MEAN_IMPUTE")

            # Skewness (only if numeric and meaningful)
            if abs(skew) > 1 and col not in ["O2Sat"]:
                decisions.append("LOG_TRANSFORM")

            # Outliers
            if outlier > 0.1:
                decisions.append("ROBUST_SCALE")

            self.decisions[col] = decisions

        return self.decisions

In [ ]:
class RepairAgent:
    def __init__(self, df, decisions):
        self.df = df.copy()
        self.decisions = decisions
        self.log = []

    def apply(self):
        for col, actions in self.decisions.items():

            if "DROP" in actions:
                self.df.drop(columns=[col], inplace=True, errors='ignore')
                self.log.append(f"Dropped {col}")
                continue

            if col not in self.df.columns:
                continue

            # Mean Imputation
            if "MEAN_IMPUTE" in actions:
                self.df[col].fillna(self.df[col].mean(), inplace=True)
                self.log.append(f"Mean imputed {col}")

            # Advanced Impute (simple median for now)
            if "ADVANCED_IMPUTE" in actions:
                self.df[col].fillna(self.df[col].median(), inplace=True)
                self.log.append(f"Median imputed {col}")

            if col == "SepsisLabel":
             continue

            if "LOG_TRANSFORM" in actions:
            # Prevent log of negative values
                self.df[col] = np.log1p(self.df[col].clip(lower=0))
                self.log.append(f"Log transformed {col} (safe)")

        return self.df

In [ ]:
repair = RepairAgent(df, decisions)
clean_df = repair.apply()

print(clean_df.head())

/tmp/ipykernel_18832/19016292.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(self.df[col].mean(), inplace=True)
/tmp/ipykernel_18832/19016292.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

   Unnamed: 0      Hour         HR     O2Sat  Temp         SBP       MAP  \
0    0.000000  0.000000  84.581443  4.586945  37.0  123.750465  4.423650   
1    0.693147  0.693147  65.000000  4.615121  37.0  123.750465  4.290459   
2    1.098612  1.098612  78.000000  4.615121  37.0  123.750465  3.772761   
3    1.386294  1.386294  73.000000  4.615121  37.0  123.750465  4.423650   
4    1.609438  1.609438  70.000000  4.615121  37.0  129.000000  4.317488   

        DBP       Resp   Glucose    Age  Gender     Unit1     Unit2  \
0  4.171777  18.726498  4.852030  68.54       0  0.496571  0.503429   
1  4.171777  16.500000  4.852030  68.54       0  0.496571  0.503429   
2  4.171777  18.726498  4.852030  68.54       0  0.496571  0.503429   
3  4.171777  17.000000  4.852030  68.54       0  0.496571  0.503429   
4  4.248495  14.000000  5.087596  68.54       0  0.496571  0.503429   

   HospAdmTime    ICULOS  SepsisLabel  Patient_ID  
0          0.0  0.693147            0       17072  
1          0

/tmp/ipykernel_18832/19016292.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(self.df[col].mean(), inplace=True)


In [ ]:
clean_df.isnull().mean().head(10)

In [ ]:
from sklearn.model_selection import train_test_split

X = clean_df.drop("SepsisLabel", axis=1)
y = clean_df["SepsisLabel"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
# Ensure label is binary
clean_df["SepsisLabel"] = clean_df["SepsisLabel"].round().astype(int)

# Check values
print(clean_df["SepsisLabel"].unique())

[0 1]


In [ ]:
print(clean_df["SepsisLabel"].dtype)
print(clean_df["SepsisLabel"].unique())

int64
[0 1]


In [ ]:
from sklearn.model_selection import train_test_split

X = clean_df.drop("SepsisLabel", axis=1)
y = clean_df["SepsisLabel"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000, class_weight='balanced')

model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print(y_pred[:20])

[0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 1 1]


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))

print("AUC Score:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.99      0.66      0.79    304859
           1       0.03      0.63      0.06      5583

    accuracy                           0.66    310442
   macro avg       0.51      0.65      0.43    310442
weighted avg       0.97      0.66      0.78    310442

AUC Score: 0.6946290501740848


In [ ]:
# =========================
# BASELINE PIPELINE
# =========================

baseline_df = df.copy()

# Drop useless column
if "Unnamed: 0" in baseline_df.columns:
    baseline_df.drop(columns=["Unnamed: 0"], inplace=True)

# Simple imputation (NO intelligence)
baseline_df = baseline_df.fillna(baseline_df.mean(numeric_only=True))

# Ensure label is correct
baseline_df["SepsisLabel"] = baseline_df["SepsisLabel"].astype(int)

# Split
from sklearn.model_selection import train_test_split

X_base = baseline_df.drop("SepsisLabel", axis=1)
y_base = baseline_df["SepsisLabel"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)

# Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_b = scaler.fit_transform(X_train_b)
X_test_b = scaler.transform(X_test_b)

# Model
from sklearn.linear_model import LogisticRegression

model_base = LogisticRegression(max_iter=2000, class_weight='balanced')
model_base.fit(X_train_b, y_train_b)

# Predict
y_pred_b = model_base.predict(X_test_b)
y_prob_b = model_base.predict_proba(X_test_b)[:,1]

# Evaluate
from sklearn.metrics import classification_report, roc_auc_score

print("BASELINE MODEL RESULTS")
print(classification_report(y_test_b, y_pred_b))
print("AUC:", roc_auc_score(y_test_b, y_prob_b))

BASELINE MODEL RESULTS
              precision    recall  f1-score   support

           0       0.99      0.75      0.85    304859
           1       0.04      0.61      0.08      5583

    accuracy                           0.75    310442
   macro avg       0.52      0.68      0.47    310442
weighted avg       0.97      0.75      0.84    310442

AUC: 0.7270971156765426


In [ ]:
import numpy as np

threshold = 0.3  # try 0.2–0.4

y_pred_new = (y_prob > threshold).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_new))

              precision    recall  f1-score   support

           0       0.99      0.17      0.29    304859
           1       0.02      0.92      0.04      5583

    accuracy                           0.19    310442
   macro avg       0.51      0.55      0.17    310442
weighted avg       0.97      0.19      0.29    310442



In [ ]:
import numpy as np
from sklearn.metrics import classification_report

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    print(f"\nThreshold: {t}")
    y_pred_t = (y_prob > t).astype(int)
    print(classification_report(y_test, y_pred_t))


Threshold: 0.2
              precision    recall  f1-score   support

           0       0.99      0.03      0.07    304859
           1       0.02      0.99      0.04      5583

    accuracy                           0.05    310442
   macro avg       0.51      0.51      0.05    310442
weighted avg       0.98      0.05      0.06    310442


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.99      0.17      0.29    304859
           1       0.02      0.92      0.04      5583

    accuracy                           0.19    310442
   macro avg       0.51      0.55      0.17    310442
weighted avg       0.97      0.19      0.29    310442


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.99      0.40      0.57    304859
           1       0.02      0.80      0.05      5583

    accuracy                           0.41    310442
   macro avg       0.51      0.60      0.31    310442
weighted avg       0.97   

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print("RANDOM FOREST RESULTS")
print(classification_report(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_prob_rf))

RANDOM FOREST RESULTS
              precision    recall  f1-score   support

           0       0.99      0.88      0.93    304859
           1       0.08      0.56      0.14      5583

    accuracy                           0.88    310442
   macro avg       0.54      0.72      0.54    310442
weighted avg       0.97      0.88      0.92    310442

AUC: 0.8201180412331421


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [ ]:
rf.fit(X_train_sm, y_train_sm)

RandomForestClassifier(class_weight='balanced_subsample', max_depth=12,
                       min_samples_split=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

best_thresh = 0
best_score = 0

for t in np.arange(0.2, 0.7, 0.05):
    y_pred_t = (y_prob_rf > t).astype(int)
    score = f1_score(y_test, y_pred_t)

    if score > best_score:
        best_score = score
        best_thresh = t

print("Best Threshold:", best_thresh)

Best Threshold: 0.6499999999999999


In [ ]:
y_pred_final = (y_prob_rf > best_thresh).astype(int)

print(classification_report(y_test, y_pred_final))
print("AUC:", roc_auc_score(y_test, y_prob_rf))

              precision    recall  f1-score   support

           0       0.99      0.96      0.97    304859
           1       0.13      0.36      0.19      5583

    accuracy                           0.95    310442
   macro avg       0.56      0.66      0.58    310442
weighted avg       0.97      0.95      0.96    310442

AUC: 0.8201180412331421


In [ ]:
def hybrid_predict(y_prob):
    predictions = []

    for p in y_prob:
        if p > 0.6:
            predictions.append(2)  # High risk
        elif p > 0.3:
            predictions.append(1)  # Medium risk
        else:
            predictions.append(0)  # Low risk

    return np.array(predictions)

In [ ]:
hybrid_preds = hybrid_predict(y_prob_rf)

In [ ]:
import numpy as np

unique, counts = np.unique(hybrid_preds, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Class {u}: {c}")

Class 0: 148992
Class 1: 142806
Class 2: 18644


In [ ]:
hybrid_binary = (hybrid_preds > 0).astype(int)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

print("HYBRID MODEL RESULTS")
print(classification_report(y_test, hybrid_binary))

print("AUC:", roc_auc_score(y_test, y_prob_rf))

HYBRID MODEL RESULTS
              precision    recall  f1-score   support

           0       1.00      0.49      0.65    304859
           1       0.03      0.90      0.06      5583

    accuracy                           0.49    310442
   macro avg       0.51      0.69      0.36    310442
weighted avg       0.98      0.49      0.64    310442

AUC: 0.8201180412331421


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# =========================
# STORE RESULTS FUNCTION
# =========================
def get_metrics(y_true, y_pred, y_prob):
    return {
        "Precision (1)": precision_score(y_true, y_pred),
        "Recall (1)": recall_score(y_true, y_pred),
        "F1-score (1)": f1_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob)
    }

# =========================
# BASELINE MODEL METRICS
# =========================
baseline_metrics = get_metrics(y_test_b, y_pred_b, y_prob_b)

# =========================
# IMPROVED AGENT MODEL (RF)
# =========================
agent_metrics = get_metrics(y_test, y_pred_rf, y_prob_rf)

# =========================
# HYBRID MODEL METRICS
# =========================
hybrid_metrics = get_metrics(y_test, hybrid_binary, y_prob_rf)

# =========================
# CREATE COMPARISON TABLE
# =========================
comparison_df = pd.DataFrame({
    "Baseline": baseline_metrics,
    "Agent (RF + SMOTE)": agent_metrics,
    "Hybrid Model": hybrid_metrics
}).T

# Round for clean display
comparison_df = comparison_df.round(3)

# Display
print("🔥 FINAL MODEL COMPARISON")
display(comparison_df)

🔥 FINAL MODEL COMPARISON


,Precision (1),Recall (1),F1-score (1),AUC
Baseline,0.042,0.606,0.079,0.727
Agent (RF + SMOTE),0.080,0.556,0.140,0.820
Hybrid Model,0.031,0.896,0.060,0.820


In [ ]:
from sklearn.metrics import accuracy_score

comparison_df["Accuracy"] = [
    accuracy_score(y_test_b, y_pred_b),
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, hybrid_binary)
]

display(comparison_df.round(3))

,Precision (1),Recall (1),F1-score (1),AUC,Accuracy
Baseline,0.042,0.606,0.079,0.727,0.745
Agent (RF + SMOTE),0.080,0.556,0.140,0.820,0.877
Hybrid Model,0.031,0.896,0.060,0.820,0.494


In [ ]:
comparison_df.to_csv("final_results.csv", index=True)

In [ ]:
print("Total missing after cleaning:", clean_df.isnull().sum().sum())

Total missing after cleaning: 0


In [ ]:
import joblib
joblib.dump(rf, "final_model.pkl")

['final_model.pkl']

In [ ]:
from google.colab import files
files.download("final_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>